In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, precision_recall_curve, average_precision_score
from sklearn.ensemble import RandomForestClassifier

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)

class DefaultRateAnalysis:
    def __init__(self, random_state=42):
        self.random_state = random_state
        
    def demonstrate_impact_of_different_rates(self, n_samples=10000):
        """
        Demonstrate the impact of different default rates in train/test sets
        """
        # Generate base dataset
        np.random.seed(self.random_state)
        df = self.generate_credit_data(n_samples)
        
        # Create splits with different methods
        splits = {
            'stratified': self.create_stratified_split(df),
            'random': self.create_random_split(df),
            'biased': self.create_biased_split(df)
        }
        
        # Analyze each split
        results = {}
        for split_name, (X_train, X_test, y_train, y_test) in splits.items():
            results[split_name] = self.analyze_split(
                X_train, X_test, y_train, y_test, split_name
            )

        return results
    
    def generate_credit_data(self, n_samples):
        """Generate synthetic credit scoring data"""
        df = pd.DataFrame({
            'income': np.random.normal(50000, 20000, n_samples),
            'debt_ratio': np.random.uniform(0.1, 0.6, n_samples),
            'credit_score': np.random.normal(700, 50, n_samples),
            'default': np.random.choice([0, 1], n_samples, p=[0.95, 0.05])
        })
        
        # Add realistic correlations
        df.loc[df['credit_score'] < 650, 'default'] = \
            np.random.choice([0, 1], sum(df['credit_score'] < 650), p=[0.8, 0.2])
        return df
    
    def create_stratified_split(self, df):
        """Create stratified train/test split"""
        return train_test_split(
            df.drop('default', axis=1),
            df['default'],
            test_size=0.2,
            stratify=df['default'],
            random_state=self.random_state
        )
    
    def create_random_split(self, df):
        """Create random (non-stratified) split"""
        return train_test_split(
            df.drop('default', axis=1),
            df['default'],
            test_size=0.2,
            stratify=None,
            random_state=self.random_state
        )
    
    def create_biased_split(self, df):
        """Create intentionally biased split"""
        # Sort by credit_score to create biased splits
        df_sorted = df.sort_values('credit_score')
        split_idx = int(len(df) * 0.8)
        
        X_train = df_sorted.iloc[:split_idx].drop('default', axis=1)
        X_test = df_sorted.iloc[split_idx:].drop('default', axis=1)
        y_train = df_sorted.iloc[:split_idx]['default']
        y_test = df_sorted.iloc[split_idx:]['default']
        
        return X_train, X_test, y_train, y_test
    
    def analyze_split(self, X_train, X_test, y_train, y_test, split_name):
        """Analyze the impact of split on model performance"""
        # Train model
        model = RandomForestClassifier(random_state=self.random_state)
        model.fit(X_train, y_train)
        
        # Get predictions
        y_pred_train = model.predict_proba(X_train)[:, 1]
        y_pred_test = model.predict_proba(X_test)[:, 1]
        
        # Calculate metrics
        results = {
            'default_rates': {
                'train': y_train.mean(),
                'test': y_test.mean(),
                'difference': abs(y_train.mean() - y_test.mean())
            },
            'performance': {
                'train_auc': roc_auc_score(y_train, y_pred_train),
                'test_auc': roc_auc_score(y_test, y_pred_test),
                'auc_difference': abs(
                    roc_auc_score(y_train, y_pred_train) - 
                    roc_auc_score(y_test, y_pred_test)
                )
            },
            'calibration': {
                'train_avg_pred': y_pred_train.mean(),
                'test_avg_pred': y_pred_test.mean(),
                'prediction_bias': y_pred_test.mean() - y_test.mean()
            }
        }
        
        return results

def run_demonstration():
    """Run demonstration and display results"""
    analyzer = DefaultRateAnalysis()
    results = analyzer.demonstrate_impact_of_different_rates()
    print("number of results is ", len(results))
    # Print results
    print("\nImpact of Different Default Rates in Train/Test Sets")
    print("=" * 50)
    
    for split_name, metrics in results.items():
        print(f"\n{split_name.capitalize()} Split:")
        print("-" * 30)
        
        # Default rates
        dr = metrics['default_rates']
        print(f"Default Rates:")
        print(f"  Train: {dr['train']:.3f}")
        print(f"  Test:  {dr['test']:.3f}")
        print(f"  Difference: {dr['difference']:.3f}")
        
        # Performance
        perf = metrics['performance']
        print(f"\nModel Performance:")
        print(f"  Train AUC: {perf['train_auc']:.3f}")
        print(f"  Test AUC:  {perf['test_auc']:.3f}")
        print(f"  AUC Difference: {perf['auc_difference']:.3f}")
        
        # Calibration
        cal = metrics['calibration']
        print(f"\nModel Calibration:")
        print(f"  Prediction Bias: {cal['prediction_bias']:.3f}")
    
    return results

# Run the demonstration
results = run_demonstration()

number of results is  3

Impact of Different Default Rates in Train/Test Sets

Stratified Split:
------------------------------
Default Rates:
  Train: 0.073
  Test:  0.073
  Difference: 0.000

Model Performance:
  Train AUC: 1.000
  Test AUC:  0.587
  AUC Difference: 0.413

Model Calibration:
  Prediction Bias: 0.003

Random Split:
------------------------------
Default Rates:
  Train: 0.073
  Test:  0.074
  Difference: 0.001

Model Performance:
  Train AUC: 1.000
  Test AUC:  0.594
  AUC Difference: 0.406

Model Calibration:
  Prediction Bias: 0.001

Biased Split:
------------------------------
Default Rates:
  Train: 0.079
  Test:  0.053
  Difference: 0.025

Model Performance:
  Train AUC: 1.000
  Test AUC:  0.548
  AUC Difference: 0.452

Model Calibration:
  Prediction Bias: 0.320
